# Differentiable SHA-256 walkthrough

This notebook verifies the exact binary forward pass, then treats a padded message block as a continuous PyTorch variable to inspect gradients through the tensor implementation. It is an experiment in differentiable programming, not a cryptanalytic claim.

In [ ]:
from hashlib import sha256

import torch

from bbtsha import DifferentiableSHA256, bits_to_bytes, bytes_to_bits, pad_single_block

## Exact binary forward pass

For a binary padded block, the tensor program matches standard SHA-256.

In [ ]:
message = "b" * 55
block, message_bits = pad_single_block(message)
expected = sha256(message.encode("utf-8")).digest()
actual_bits = DifferentiableSHA256().hash_block(block)

assert bits_to_bytes(actual_bits) == expected
expected.hex()

## Autograd experiment

The exactness guarantee is only for binary inputs. Allowing the message bits to vary continuously creates a differentiable relaxation that can be used to inspect gradients.

In [ ]:
relaxed_block = block.clone().detach().requires_grad_(True)
relaxed_digest = DifferentiableSHA256().hash_block(relaxed_block)
objective = relaxed_digest.sum()
objective.backward()

{
    "objective": objective.item(),
    "finite_gradients": torch.isfinite(relaxed_block.grad).all().item(),
    "message_gradient_norm": relaxed_block.grad[:message_bits].norm().item(),
}